# Guide intelligent du Musée · 05 Questions (jalon `m4`)

Seconde fonction du guide : proposer les œuvres qui répondent à la question d'un visiteur. Un modèle classique (TF-IDF) puis un modèle neuronal (embeddings de phrases), mesurés sur le même jeu de questions.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from commun import UNKNOWN_ARTIST, load_metadata, set_seed

set_seed()
meta, CLASSES = load_metadata()
print(f"{len(meta)} œuvres, {len(CLASSES)} styles")

## 2. Descriptions des œuvres et questions de test

Les métadonnées décrivent mal le contenu des œuvres : le genre est souvent inconnu, la plupart des artistes aussi, et rien ne dit ce que montre une image.

In [ ]:
known_genre = meta["genre"] != "Unknown Genre"
named = meta["artist"] != UNKNOWN_ARTIST
print(f"Genre connu pour {known_genre.sum()} œuvres sur {len(meta)} : "
      + ", ".join(f"{g} {n}" for g, n in meta.loc[known_genre, "genre"].value_counts().items()))
print(f"Artiste nommé pour {named.sum()} œuvres, {meta.loc[named, 'artist'].nunique()} artistes différents")

Chaque description reprend donc les métadonnées connues (style, genre, artiste) et les complète d'une courte présentation du mouvement et du genre, écrite à la main comme un cartel de musée. Ces présentations ont été écrites avant les questions et ne citent aucun artiste : les noms d'artistes ne viennent que des métadonnées.

Les descriptions et les questions sont en anglais, comme les exemples des consignes, car le modèle d'embeddings de la section 4 (`all-MiniLM-L6-v2`) est entraîné sur de l'anglais. Un guide en français demanderait un modèle multilingue.

### 2.1 Descriptions

In [ ]:
STYLE_NAMES = {"Baroque": "Baroque", "Cubism": "Cubist", "Impressionism": "Impressionist",
               "Pop_Art": "Pop Art", "Ukiyo_e": "Ukiyo-e"}
MOVEMENTS = {
    "Baroque": "Baroque is the art of 17th-century Europe, in Italy, Spain, Flanders and the Netherlands. "
               "Painters used dramatic contrasts of light and shadow, rich dark colours and movement to stage "
               "religious scenes, mythology and portraits of the powerful, in oil on canvas.",
    "Cubism": "Cubism is an early 20th-century movement born in Paris. It breaks objects and figures into "
              "geometric facets seen from several viewpoints at once, often in muted browns, greys and ochres, "
              "and sometimes adds collage with newspaper or wallpaper.",
    "Impressionism": "Impressionism is a French movement of the late 19th century. Painters worked outdoors to "
                     "capture fleeting light, weather and atmosphere with quick, visible brushstrokes and bright "
                     "colours, painting landscapes, rivers, gardens and scenes of modern Parisian life.",
    "Pop_Art": "Pop Art is a movement of the 1950s and 1960s in Britain and the United States. Artists borrowed "
               "images from advertising, comic strips, packaging and film stars, using flat bright colours, bold "
               "outlines and commercial techniques such as screen printing.",
    "Ukiyo_e": "Ukiyo-e, pictures of the floating world, are Japanese woodblock prints of the Edo period, from "
               "the 17th to the 19th century. They depict kabuki actors, beautiful women, landscapes, Mount Fuji "
               "and waves with elegant outlines and flat areas of colour.",
}
GENRE_NAMES = {"still_life": "still life", "genre_painting": "genre scene", "sketch_and_study": "sketch",
               "landscape": "landscape"}
GENRES = {
    "still_life": "A still life shows an arrangement of objects such as fruit, flowers, bottles, glasses, jugs "
                  "or musical instruments.",
    "genre_painting": "A genre scene shows ordinary people in everyday life, at work, at home or at leisure.",
    "sketch_and_study": "A sketch or study is a quick preparatory drawing or painting made before a finished work.",
    "landscape": "A landscape shows a view of nature, such as countryside, mountains, rivers or the sea.",
}


def describe(row) -> str:
    """Description d'une œuvre : style, genre et artiste connus, puis présentation du mouvement et du genre."""
    head = f"{STYLE_NAMES[row.label]} {GENRE_NAMES.get(row.genre, 'work')}"
    if row.artist != UNKNOWN_ARTIST:
        head += " by " + row.artist.replace("-", " ").title()
    return " ".join([head + ".", MOVEMENTS[row.label], GENRES.get(row.genre, "")]).strip()


descriptions = [describe(row) for row in meta.itertuples()]
print(f"{len(set(descriptions))} descriptions distinctes pour {len(descriptions)} œuvres, par exemple :\n")
for i in (meta.index[meta["artist"] == "georges-braque"][0], meta.index[meta["label"] == "Impressionism"][0]):
    print(descriptions[i], "\n")

### 2.2 Questions de test

45 questions écrites à la main, chacune avec ses réponses attendues : les œuvres d'un style, d'un style et d'un genre, ou d'un artiste. Chaque question est classée automatiquement :

- **mots partagés** si elle a au moins un mot en commun avec la description d'une œuvre attendue, après le même découpage en mots que TF-IDF (minuscules, mots vides anglais retirés) ;
- **paraphrase** sinon : allusions (« art of the Dutch Golden Age »), artistes célèbres absents des métadonnées (Picasso, Monet), synonymes.

Deux questions des consignes restent hors du score, faute de réponse attendue : les métadonnées ne disent pas quelle œuvre montre une mer déchaînée ou des enfants.

In [ ]:
QUESTIONS = [
    ("Show me Baroque paintings", {"style": "Baroque"}),
    ("I would like to see Cubist art", {"style": "Cubism"}),
    ("Impressionist paintings", {"style": "Impressionism"}),
    ("Pop Art works", {"style": "Pop_Art"}),
    ("Japanese woodblock prints", {"style": "Ukiyo_e"}),
    ("paintings with dramatic contrasts of light and shadow", {"style": "Baroque"}),
    ("religious scenes from the 1600s", {"style": "Baroque"}),
    ("theatrical Old Master paintings", {"style": "Baroque"}),
    ("objects broken into geometric shapes", {"style": "Cubism"}),
    ("a subject seen from many angles at the same time", {"style": "Cubism"}),
    ("modern art from Paris around 1910", {"style": "Cubism"}),
    ("paintings made outdoors to catch the sunlight", {"style": "Impressionism"}),
    ("French art from the 1870s with loose brushwork", {"style": "Impressionism"}),
    ("the changing effects of daylight on water", {"style": "Impressionism"}),
    ("art inspired by comics and advertising", {"style": "Pop_Art"}),
    ("pictures of movie stars and supermarket products", {"style": "Pop_Art"}),
    ("American art about consumer culture in the sixties", {"style": "Pop_Art"}),
    ("views of Mount Fuji", {"style": "Ukiyo_e"}),
    ("portraits of kabuki actors", {"style": "Ukiyo_e"}),
    ("pictures of the floating world", {"style": "Ukiyo_e"}),
    ("traditional prints from Edo-period Japan", {"style": "Ukiyo_e"}),
    ("Cubist still lifes", {"style": "Cubism", "genre": "still_life"}),
    ("a still life with a guitar and a bottle", {"genre": "still_life"}),
    ("fruit and flowers on a table from the 17th century", {"style": "Baroque", "genre": "still_life"}),
    ("scenes of daily life in old Japan", {"style": "Ukiyo_e", "genre": "genre_painting"}),
    ("preparatory drawings by Baroque masters", {"style": "Baroque", "genre": "sketch_and_study"}),
    ("works by Hokusai", {"artist": "katsushika-hokusai"}),
    ("paintings by Georges Braque", {"artist": "georges-braque"}),
    ("Rubens", {"artist": "peter-paul-rubens"}),
    ("prints by Kuniyoshi", {"artist": "utagawa-kuniyoshi"}),
    ("Juan Gris", {"artist": "juan-gris"}),
    ("Fernand Léger", {"artist": "fernand-leger"}),
    ("Murillo", {"artist": "bartolome-esteban-murillo"}),
    ("art of the Dutch Golden Age", {"style": "Baroque"}),
    ("candlelit saints and martyrs", {"style": "Baroque"}),
    ("Rembrandt and Caravaggio", {"style": "Baroque"}),
    ("fragmented, angular forms", {"style": "Cubism"}),
    ("Picasso", {"style": "Cubism"}),
    ("hazy sunsets and blurry reflections", {"style": "Impressionism"}),
    ("Monet and Renoir", {"style": "Impressionism"}),
    ("Warhol soup cans", {"style": "Pop_Art"}),
    ("Andy Warhol and Roy Lichtenstein", {"style": "Pop_Art"}),
    ("geisha and samurai", {"style": "Ukiyo_e"}),
    ("the Great Wave off Kanagawa", {"style": "Ukiyo_e"}),
    ("Hiroshige", {"style": "Ukiyo_e"}),
]
OUT_OF_SCOPE = ["Which painting shows a stormy sea?", "I would like to see portraits of children"]

analyze = TfidfVectorizer(stop_words="english").build_analyzer()  # découpage en mots de TF-IDF
COLUMNS = {"style": "label", "genre": "genre", "artist": "artist"}


def expected_mask(expected: dict[str, str]) -> np.ndarray:
    """Œuvres attendues pour une question : celles qui vérifient tous les critères."""
    mask = np.ones(len(meta), dtype=bool)
    for key, value in expected.items():
        mask &= (meta[COLUMNS[key]] == value).to_numpy()
    return mask


rows = []
for question, expected in QUESTIONS:
    mask = expected_mask(expected)
    expected_words = set().union(*(analyze(descriptions[i]) for i in np.flatnonzero(mask)))
    rows.append({"question": question, "attendu": " + ".join(expected).replace("artist", "artiste"), "oeuvres_attendues": int(mask.sum()),
                 "type": "mots partagés" if set(analyze(question)) & expected_words else "paraphrase"})
questions = pd.DataFrame(rows)
print(f"{len(questions)} questions : " + ", ".join(f"{t} {n}" for t, n in questions["type"].value_counts().items()))
pd.crosstab(questions["attendu"], questions["type"])

### 2.3 Mesure

Pour chaque question, une méthode donne un score à chacune des 1 500 œuvres et les classe par score décroissant ; les œuvres de même description, donc de même score, gardent l'ordre de la collection. On note le rang de la première œuvre attendue, puis la part des questions dont une œuvre attendue figure parmi les k premières proposées, pour k = 1, 5 et 10. Une borne affiche quelques œuvres : k = 5 est la mesure principale. Si toutes les œuvres ont un score nul, la méthode ne propose rien et la question compte comme un échec.

In [ ]:
K_VALUES = [1, 5, 10]
masks = [expected_mask(expected) for _, expected in QUESTIONS]


def evaluate(score_fn) -> pd.DataFrame:
    """Rang de la première œuvre attendue pour chaque question (NaN si la méthode ne propose rien)."""
    ranks = []
    for (question, _), mask in zip(QUESTIONS, masks):
        scores = score_fn(question)
        order = np.argsort(-scores, kind="stable")
        ranks.append(np.flatnonzero(mask[order])[0] + 1 if scores.max() > 0 else np.nan)
    results = questions.assign(rang=ranks)
    for k in K_VALUES:
        results[f"top_{k}"] = results["rang"] <= k
    return results


def hit_rates(results: pd.DataFrame) -> pd.DataFrame:
    """Part des questions avec une œuvre attendue parmi les k premières, par type de question et au total."""
    columns = [f"top_{k}" for k in K_VALUES]
    by_type = results.groupby("type")[columns].mean()
    by_type.loc["toutes"] = results[columns].mean()
    return by_type.assign(questions=results.groupby("type").size().reindex(by_type.index, fill_value=len(results)))


def top_works(score_fn, question: str, n: int = 3) -> pd.DataFrame:
    """Les n premières œuvres proposées pour une question, avec leur score."""
    scores = score_fn(question)
    order = np.argsort(-scores, kind="stable")[:n]
    return meta.loc[order, ["label", "genre", "artist"]].assign(score=scores[order].round(3)).reset_index(drop=True)

## 3. TF-IDF

Modèle classique de comparaison : chaque texte devient un vecteur TF-IDF (fréquence d'un mot dans le texte, pondérée par sa rareté dans l'ensemble des descriptions), avec le `TfidfVectorizer` de scikit-learn, mots vides anglais retirés et réglages par défaut. Aucun réglage n'est choisi sur les questions. Le score d'une œuvre est la similarité cosinus entre les vecteurs de la question et de sa description.

### 3.1 Score sur le jeu de questions

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(descriptions)  # vecteurs normalisés : produit scalaire = cosinus


def tfidf_scores(question: str) -> np.ndarray:
    return (doc_vectors @ vectorizer.transform([question]).T).toarray().ravel()


tfidf_results = evaluate(tfidf_scores)
print(f"Vocabulaire : {len(vectorizer.vocabulary_)} mots")
print(f"Questions sans aucune proposition : {tfidf_results['rang'].isna().sum()}")
hit_rates(tfidf_results).round(3)

- **TF-IDF répond bien quand la question reprend les mots des descriptions** : pour 0,875 des questions à mots partagés, la première œuvre proposée est attendue.
- **Il ne répond à aucune paraphrase** : sans mot commun, toutes les similarités sont nulles et il ne propose rien, pour 13 questions. Il ne sait pas que Monet est impressionniste, ni qu'un samouraï évoque le Japon.
- **Top 1, top 5 et top 10 sont identiques** : les œuvres de même description ont le même score. La première œuvre proposée est donc soit attendue, soit en tête d'un bloc de mauvaises réponses de même description, long de dizaines ou de centaines d'œuvres.

### 3.2 Exemples d'échecs

Les questions dont aucune œuvre attendue n'est dans les 5 premières, avec le rang de la première œuvre attendue, puis les 3 œuvres proposées en tête pour quelques-unes d'entre elles et pour les deux questions hors score.

In [ ]:
failures = tfidf_results.loc[~tfidf_results["top_5"], ["question", "attendu", "type", "rang"]]
print(f"{len(failures)} échecs sur {len(tfidf_results)} questions")
failures

In [ ]:
for question in ["Cubist still lifes", "scenes of daily life in old Japan",
                 "pictures of movie stars and supermarket products", "art of the Dutch Golden Age", *OUT_OF_SCOPE]:
    print(question)
    print(top_works(tfidf_scores, question).to_string(), "\n")

- **Les mots doivent être identiques** : « lifes » ne correspond pas à « life », ni « Japan » à « Japanese ». Pour « Cubist still lifes », seul « cubist » compte. TF-IDF propose bien des œuvres cubistes, mais d'abord celles sans genre, dont la description plus courte donne un cosinus plus fort : la première nature morte cubiste n'arrive qu'au rang 180.
- **Un mot commun peut tromper** : dans « scenes of daily life in old Japan », « life » renvoie aux natures mortes (« still life ») baroques. Dans « art of the Dutch Golden Age », « art » renvoie au Pop Art. Dans « pictures of movie stars and supermarket products », « pictures » renvoie à l'Ukiyo-e (« pictures of the floating world ») avant que « stars » ne renvoie au Pop Art.
- **Les questions hors score montrent la limite des métadonnées** : pour la mer déchaînée, TF-IDF propose des paysages, parce que la présentation du genre paysage cite la mer, pas parce qu'une œuvre montre une mer. Pour les portraits d'enfants, il propose du Baroque, dont la présentation cite les portraits. Y répondre vraiment demanderait de décrire le contenu de chaque image.

### 3.3 Bilan

- **TF-IDF : 0,622 des questions ont une œuvre attendue parmi les 5 premières proposées**, 0,875 pour les questions à mots partagés et aucune pour les paraphrases.
- **Il est rapide, transparent et sans modèle à télécharger**, mais il ne voit que les mots exacts : ni synonymes, ni pluriels irréguliers, ni connaissances sur l'art (Picasso, Monet, Warhol).
- **Le score des questions à mots partagés est sans doute optimiste** : l'équipe a écrit à la fois les descriptions et les questions.
- La section 4 pose les mêmes questions à un modèle d'embeddings de phrases, qui compare des sens plutôt que des mots.

## 4. Embeddings de phrases

- `all-MiniLM-L6-v2` pour les questions et les descriptions, puis l'œuvre de plus forte similarité cosinus.

In [ ]:
# À compléter

## 5. Comparaison

- Score sur le jeu de questions, temps d'encodage sur CPU, taille du modèle.
- Questions où chaque méthode échoue.

In [ ]:
# À compléter